# Speaker diarization

Who spoke when. This notebook runs pyannote's Community-1 pipeline on one minute of a Thai meeting, shows and plays its
turns, shows what Voxdiar would send Voxpick, saves the turns as RTTM, scores them against two references and tries the
main settings. It is the code of the manual's *Speaker Diarization* section, Steps 1 to 10; the manual explains what each
output means and where the model fails.

**Before you start** (details in the manual, *Getting started*):

- A Hugging Face token, and the conditions of
  [`pyannote/speaker-diarization-community-1`](https://huggingface.co/pyannote/speaker-diarization-community-1) accepted
  once for your account. Never paste the token into a cell.
- **Colab:** *Runtime → Change runtime type → T4 GPU*, add the token as a secret named `HF_TOKEN` with notebook access
  on, and upload `data/` with the Files panel.
- **Locally:** `uv sync`, and start Jupyter from an environment where `HF_TOKEN` is set. Skip Step 1.

The example minute is `data/lotusdis-S075/` (LOTUSDIS, CC BY-SA 4.0; see the README). Every expected output below comes
from one Colab run on a T4 (torch 2.9.1) on 2026-11-11. A Mac run on CPU the same day gave the same numbers and the same
RTTM, byte for byte; only Step 4's time differed. On another machine the numbers can move a little.

## 1. Install (Colab only)

Locally, skip this cell: `uv sync` installed the same versions. In Colab, run it, then *Runtime → Restart session*, so
that the next steps load these versions.

In Colab, pip prints a red `ERROR` with a list of dependency conflicts: Colab's own Google Cloud, TensorFlow and ADK
packages want a newer `protobuf` and `opentelemetry` than pyannote.audio 4.0.0 allows. This notebook uses none of them,
and every step ran (2026-11-11).

In [ ]:
# Step 1. install the versions in this repository's uv.lock (Colab only)
!pip install -q pyannote.audio==4.0.0 torch==2.9.1 torchaudio==2.9.1 torchvision==0.24.1 torchcodec==0.9.1 pyannote.metrics==4.1

## 2. Versions, device and token

In [ ]:
# Step 2. versions, device, token
import os
os.environ["PYANNOTE_METRICS_ENABLED"] = "0"  # pyannote.audio's usage telemetry off

import numpy as np
import torch
from importlib.metadata import version

assert np.__version__ == version("numpy"), "numpy changed on disk, restart the session"
for pkg in ["torch", "pyannote.audio", "pyannote.core", "pyannote.metrics"]:
    print(pkg, version(pkg))

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", DEVICE)

try:
    from google.colab import files, userdata  # Colab: the secret named HF_TOKEN
    IN_COLAB, HF_TOKEN = True, userdata.get("HF_TOKEN")
except ImportError:  # locally: from the environment Jupyter started in
    IN_COLAB, HF_TOKEN = False, os.environ.get("HF_TOKEN")
assert HF_TOKEN, "no HF_TOKEN: see Before you start"

Expected output:

```
torch 2.9.1
pyannote.audio 4.0.0
pyannote.core 6.1.0
pyannote.metrics 4.1
device: cuda
```

`device: cpu` in Colab means the runtime has no GPU; on a machine without one it is expected, only slower. The numpy check catches Colab keeping the old numpy after Step 1: restart the session.

## 3. Load the minute

For your own audio, change `AUDIO_PATH`, and set `REFERENCE_PATH = None` if it has no reference.

In [ ]:
# Step 3. load the minute, and its reference if you have one
import soundfile as sf
from IPython.display import Audio, display

AUDIO_PATH = "data/lotusdis-S075/S075_300-360.wav"  # your own audio: change this
AUDIO_DIR = os.path.dirname(AUDIO_PATH)  # Step 7 writes its RTTM here
REFERENCE_PATH = os.path.join(AUDIO_DIR, "S075_300-360.reference.rttm")  # no reference: REFERENCE_PATH = None
wav_name = os.path.basename(AUDIO_PATH)

x, sr = sf.read(AUDIO_PATH, dtype="float32", always_2d=True)
waveform = torch.tensor(x.T)  # pyannote wants (channel, time)
duration = waveform.shape[1] / sr
print(wav_name, round(duration, 1), "s,", sr, "Hz,", waveform.shape[0], "channel")
display(Audio(x.mean(1), rate=sr))

Expected output:

```
S075_300-360.wav 60.0 s, 24000 Hz, 1 channel
```

The minute is cut sample-exact from the corpus and not resampled: pyannote resamples to 16 kHz itself.

## 4. Run Community-1

In [ ]:
# Step 4. run Community-1
import time
from pyannote.audio import Pipeline
from pyannote.audio.core.task import Problem, Resolution, Specifications

# torch >= 2.6 loads only the classes it is told are safe; these four are pyannote's own, stored in its checkpoints
torch.serialization.add_safe_globals([Specifications, Problem, Resolution, torch.torch_version.TorchVersion])

pipe = Pipeline.from_pretrained("pyannote/speaker-diarization-community-1", token=HF_TOKEN)
if pipe is None:
    raise SystemExit("accept the model's conditions on huggingface.co first")
pipe.to(DEVICE)
t0 = time.time()
out = pipe({"waveform": waveform, "sample_rate": sr})
pipelines, outputs = {"Community-1": pipe}, {"Community-1": out.speaker_diarization}
print(f"{time.time() - t0:.1f} s,", len(out.speaker_diarization.labels()), "speakers,",
      len(list(out.speaker_diarization)), "turns,", len(list(out.exclusive_speaker_diarization)), "exclusive turns")

Expected output:

```
4.3 s, 3 speakers, 31 turns, 26 exclusive turns
```

Three labels, as many as the meeting has people. The exclusive output keeps one speaker at a time, so it has fewer
turns. The time depends on the machine and its load.

- `accept the model's conditions`, or `401 Unauthorized`: see the manual's Troubleshooting.
- On a GPU it also prints a `ReproducibilityWarning`: pyannote turns TF32 off. Leave it off.
- On a Mac with FFmpeg 9 (Homebrew) it also printed `torchcodec is not installed correctly` (torchcodec 0.9 supports
  FFmpeg 4 to 8) and a `UserWarning` about `std()`. This notebook passes the audio in memory, so the run finished normally.
- **Do not print the pipeline object:** on 2026-10-28 that printed the Hugging Face token, which was then revoked.

## 5. Look and listen

In [ ]:
# Step 5. plot the reference (if there is one) above the output
import matplotlib.pyplot as plt
from pyannote.core import Annotation, Segment, notebook

rows = list(outputs.items())
if REFERENCE_PATH:
    ref = Annotation(uri=wav_name)
    for i, line in enumerate(open(REFERENCE_PATH)):
        f = line.split()
        ref[Segment(float(f[3]), float(f[3]) + float(f[4])), i] = f[7]  # one track per line, or overlaps get merged
    rows = [("reference", ref)] + rows

notebook.crop = Segment(0, duration)
fig, axes = plt.subplots(len(rows), 1, figsize=(12, 2 * len(rows)), sharex=True, squeeze=False)
for ax, (title, ann) in zip(axes[:, 0], rows):
    notebook.plot_annotation(ann, ax=ax, time=True, legend=True)
    ax.set_title(title, loc="right")
plt.tight_layout()
plt.show()

Expected: two rows, the corpus reference (M35, M36, M37) above Community-1 (SPEAKER_00 to SPEAKER_02), as in the
manual's Step 5 figure. Play the minute under Step 3 and listen at 15 to 24 s: the reference has two people talking
throughout; Community-1 has one at a time. A row's colours say nothing about another row's.

**Give each RTTM line its own track,** as this step does with `i`. In pyannote, two turns over exactly the same stretch
overwrite each other otherwise.

## 6. See what Voxdiar would send Voxpick

Downloads Voxdiar's own overlap code, pinned to one Voxglobal commit, then runs it on the minute and on two short clips
cut around one overlap, each diarized on its own as Voxdiar does.

In [ ]:
# Step 6. what Voxdiar would send Voxpick, for the minute and for two short clips (Voxglobal commit 429e148)
import urllib.request

OVERLAP_URL = ("https://raw.githubusercontent.com/thirawarit/voxglobal/"
               "429e1488fae1b60dcf9605c3d300a938ba080375/voxdiar/src/voxdiar/overlap.py")
if not os.path.exists("overlap.py"):
    urllib.request.urlretrieve(OVERLAP_URL, "overlap.py")
from overlap import analyze_turns

def voxdiar(diar, seconds):
    turns = [(label, seg.start, seg.end) for seg, _, label in diar.itertracks(yield_label=True)]
    a = analyze_turns(turns, duration_seconds=seconds, min_overlap_seconds=0.1)  # voxdiar/configs/default.yaml
    return (f"speakerCount {a.speaker_count}, overlap {a.overlap_seconds:.2f} s, overlapRatio {a.overlap_ratio:.4f}"
            f" | Multiple speakers: {a.speaker_count > 1}, Overlapping speech: {a.overlap_ratio > 0.02}")

diar = outputs["Community-1"]
print("the minute:", voxdiar(diar, duration))
for start, end in [(52.0, 55.0), (51.5, 55.5)]:
    clip = waveform[:, int(start * sr):int(end * sr)]
    clip_out = pipe({"waveform": clip, "sample_rate": sr}).speaker_diarization
    print(f"clip {start}-{end} s:", voxdiar(clip_out, clip.shape[1] / sr))

Expected output:

```
the minute: speakerCount 3, overlap 0.91 s, overlapRatio 0.0152 | Multiple speakers: True, Overlapping speech: False
clip 52.0-55.0 s: speakerCount 1, overlap 0.00 s, overlapRatio 0.0000 | Multiple speakers: False, Overlapping speech: False
clip 51.5-55.5 s: speakerCount 2, overlap 0.19 s, overlapRatio 0.0464 | Multiple speakers: True, Overlapping speech: True
```

The two thresholds are Voxpick's defaults (`voxpick/src/types.ts`). The 3 s clip holds 1.50 s of overlap by our hand
labels, yet comes back with one speaker and no tag; cut half a second wider, it gets both tags. On a short clip, a tag
can depend on where the clip was cut: listen before you trust it.

## 7. Save the result

In [ ]:
# Step 7. save the result as RTTM next to the audio, and download it in Colab
diar.uri = wav_name[:-4]  # the file field of every line; without it pyannote writes <NA>
rttm_out = os.path.join(AUDIO_DIR, wav_name[:-4] + ".community-1.rttm")
with open(rttm_out, "w") as f:
    diar.write_rttm(f)
print(*open(rttm_out).read().splitlines()[:3], sep="\n")
if IN_COLAB:
    files.download(rttm_out)

Expected output:

```
SPEAKER S075_300-360 1 0.031 1.316 <NA> <NA> SPEAKER_00 <NA> <NA>
SPEAKER S075_300-360 1 1.685 0.607 <NA> <NA> SPEAKER_01 <NA> <NA>
SPEAKER S075_300-360 1 2.697 1.367 <NA> <NA> SPEAKER_01 <NA> <NA>
```

One turn per line: field 2 is the file, 4 the start and 5 the length in seconds (the first turn ends at
0.031 + 1.316 = 1.347 s), and 8 the label.

# Optional: score against a reference

Skip this part unless you have a reference: who spoke when, labelled by a person or shipped with a corpus. Run Steps 2
to 5 first. The manual's *Evaluation* section explains the metric.

## 8. Against the corpus reference

In [ ]:
# Step 8. score against the corpus reference (collar 0, overlap counted)
from pyannote.core import Timeline
from pyannote.metrics.diarization import DiarizationErrorRate

def score(reference, name):
    for short, hyp in outputs.items():
        metric = DiarizationErrorRate(collar=0.0, skip_overlap=False)
        d = metric(reference, hyp, uem=Timeline([Segment(0, duration)]), detailed=True)
        total = d["total"]
        print(f"{short} vs {name}: DER {d['diarization error rate']:.2%}, false alarm {d['false alarm'] / total:.2%},"
              f" missed {d['missed detection'] / total:.2%}, confusion {d['confusion'] / total:.2%},"
              f" reference speech {total:.1f} s")
        print("   labels matched:", metric.optimal_mapping(reference, hyp))

score(ref, "the corpus")

Expected output:

```
Community-1 vs the corpus: DER 43.72%, false alarm 0.00%, missed 42.11%, confusion 1.60%, reference speech 91.5 s
   labels matched: {'SPEAKER_00': 'M36', 'SPEAKER_01': 'M37', 'SPEAKER_02': 'M35'}
```

Almost all of the error is missed speech: time the reference gives to a second or third person. The minute has 91.5 s of reference speech because overlap counts twice.

## 9. Against your own labels

Label the whole recording in Audacity and export the label track as a text file (start, end and name, separated by
tabs); time outside your labels counts as nobody talking. The manual's *Score it against a reference* shows how, and why
the labelling rules change the score. Here, our hand labels for the minute.

In [ ]:
# Step 9. score against your own Audacity labels
LABELS_PATH = os.path.join(AUDIO_DIR, "S075_300-360.hand-labels.txt")  # your Audacity export
mine = Annotation(uri="labels")
for i, line in enumerate(open(LABELS_PATH)):
    if line.startswith("\\"):  # Audacity adds these for spectral selections
        continue
    start, end, speaker = line.rstrip("\n").split("\t")
    mine[Segment(float(start), float(end)), i] = speaker

score(mine, "your labels")

Expected output:

```
Community-1 vs your labels: DER 20.94%, false alarm 0.10%, missed 18.40%, confusion 2.45%, reference speech 64.8 s
   labels matched: {'SPEAKER_00': 'Spk_A', 'SPEAKER_01': 'Spk_B', 'SPEAKER_02': 'Spk_C'}
```

Missed speech falls from about 42% to about 18% on the same output. **Score against labels whose rules you know.**

# Tuning

## 10. Settings, speaker counts, one clustering setting

Prints the pipeline's settings, runs three speaker counts and one changed clustering setting, then puts the defaults
back. The manual's *Tuning* section says what each setting acts on.

In [ ]:
# Step 10. Tuning: the pipeline's settings, then speaker counts, then one clustering setting, then the defaults again
import warnings

def speakers(pipe, **count):
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter("always")
        labels = pipe({"waveform": waveform, "sample_rate": sr}, **count).speaker_diarization.labels()
    note = " (warned: outside the bounds)" if any("outside" in str(w.message) for w in caught) else ""
    return f"{len(labels)} speakers{note}"

for short, pipe in pipelines.items():
    defaults = pipe.parameters(instantiated=True)
    print(short)
    print("  parameters:", defaults)
    print("  segmentation_step:", pipe.segmentation_step, "| embedding_exclude_overlap:", pipe.embedding_exclude_overlap,
          "| batch sizes (segmentation, embedding):", pipe.segmentation_batch_size, pipe.embedding_batch_size)
    print("  default:            ", speakers(pipe))
    print("  num_speakers=2:     ", speakers(pipe, num_speakers=2))
    print("  min_speakers=4:     ", speakers(pipe, min_speakers=4))
    change = {"clustering": {"threshold": 0.8}}
    pipe.instantiate(change)
    print(f"  {change}:", speakers(pipe))
    pipe.instantiate(defaults)  # back to the defaults, so later runs use them
    print("  defaults again:     ", speakers(pipe))

Expected output:

```
Community-1
  parameters: {'segmentation': {'min_duration_off': 0.0}, 'clustering': {'threshold': 0.6, 'Fa': 0.07, 'Fb': 0.8}}
  segmentation_step: 0.1 | embedding_exclude_overlap: True | batch sizes (segmentation, embedding): 32 32
  default:             3 speakers
  num_speakers=2:      2 speakers
  min_speakers=4:      4 speakers
  {'clustering': {'threshold': 0.8}}: 3 speakers
  defaults again:      3 speakers
```

`min_speakers=4` gave four labels on a three-speaker minute, with no warning. **Pass a count only when you know it.**